<hr />

### Patient Readmission Risk Predictor 

<hr />

<p>Applied Artificial Intelligence</p>

<p>Prof. Maryam Abbasi</p>

<p>Escola Superior de Gestão e Tecnologia - Instituto Politécnico de Santarém</p>

<hr />

#### Imports Install

In [1]:
!pip install scikit-learn pandas matplotlib

#### Loading Imports and Librarys

In [2]:
# ── Environment Setup ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibility
np.random.seed(42)

print(" All libraries imported successfully!")
print(f"   numpy  : {np.__version__}")
print(f"   pandas : {pd.__version__}")

 All libraries imported successfully!
   numpy  : 2.2.5
   pandas : 2.3.3


#### Starting to Separate All the Dataset, Loading the "mapping" Dataset to Get All Columns Associated with Main Data Dataset

#### After That, We Need to Divide in "Blocks" for a Better Distinguish and Convert Them to Numeric to Get The Real Value from the String Value in the "mapping" Dataset

#### At the End, We Gonna Load All Data Dataset by Chuncks Because We Don't Know How Much Size It Has and Also for Computer Resources Performance It's Much Better

In [4]:
## Datasets
path_prrp_data = "data/diabetic_data.csv"
path_prrp_map = "data/IDS_mapping.csv"

## Arguments
chunck_size = 1000
prrp_map = pd.read_csv(path_prrp_map, header=None)
prrp_df = []

## Find Empty Rows
empty_rows = prrp_map[prrp_map.isna().all(axis=1)].index.tolist()

## Separate Between 3 Blocks - admission_type, discharge_disposition, admission_source
b1 = prrp_map.iloc[0:empty_rows[0], :] ## 1st Block - Row 0 to First Empty Row
b2 = prrp_map.iloc[empty_rows[0]+1:empty_rows[1], :] ## 2nd Block - Between First and Second Empty Rows
b3 = prrp_map.iloc[empty_rows[1]+1:, :] ## 3rd Block - After Second Empty Row to the End

## Assing Correct Header to Each Block
b1.columns, b2.columns, b3.columns = ["admission_type_id", "description"], ["discharge_disposition_id", "description"], ["admission_source_id", "description"]

## Convert ID for Numeric (Ignore Empty Rows and NULL)
b1["admission_type_id"] = pd.to_numeric(b1["admission_type_id"], errors="coerce").astype("Int64")
b2["discharge_disposition_id"] = pd.to_numeric(b2["discharge_disposition_id"], errors="coerce").astype("Int64")
b3["admission_source_id"] = pd.to_numeric(b3["admission_source_id"], errors="coerce").astype("Int64")

## Rename Column "description" to Distinct Names
b1 = b1.rename(columns={"description": "admission_type"})
b2 = b2.rename(columns={"description": "discharge_disposition"})
b3 = b3.rename(columns={"description": "admission_source"})

## Process All Dataset Chunck by Chunck (Better Resources Performance)
prrp_chuncks = []
for chunck in pd.read_csv(path_prrp_data, chunksize=chunck_size):
    for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        chunck[col] = pd.to_numeric(chunck[col], errors="coerce").astype("Int64")

    chunck = chunck.merge(b1, on="admission_type_id", how="left")
    chunck = chunck.merge(b2, on="discharge_disposition_id", how="left")
    chunck = chunck.merge(b3, on="admission_source_id", how="left")
    prrp_chuncks.append(chunck)

prrp_df = pd.concat(prrp_chuncks, ignore_index=True)

#### EDA Progress

In [5]:
## General Visualization
print(prrp_df.shape)
print(prrp_df.info())

## Numerical Statistics
print(prrp_df.describe().T)

## Categorical Statistics
print(prrp_df.describe(include="object").T)

## Missing by Column (Feature)
missing = prrp_df.isna().sum().sort_values(ascending=False)
print(missing)

## Target Variable Distribution - "readmitted"
print(prrp_df["readmitted"].value_counts(dropna=False))
print(prrp_df["readmitted"].value_counts(normalize=True))

## New Column with Readmission Binary (<30)
prrp_df["readmitted_30"] = prrp_df["readmitted"].eq("<30").astype(int)
print(prrp_df["readmitted_30"].value_counts(normalize=True))

## Already Decoded Admission Type Distribution
print(prrp_df["admission_type"].value_counts(dropna=False))
print(prrp_df["discharge_disposition"].value_counts(dropna=False))
print(prrp_df["admission_source"].value_counts(dropna=False))

(101766, 53)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 53 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  Int64 
 7   discharge_disposition_id  101766 non-null  Int64 
 8   admission_source_id       101766 non-null  Int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int